# Notebook 07 — Graph Neural Networks

**Module:** 14 — Deep Learning and GNNs  
**Tier:** 2 — Working competence  
**Notebook:** 7 of 12  
**Time estimate:** 90 minutes

> Biology is fundamentally a graph problem. Proteins interact in PPI networks.
> Metabolites flow through metabolic networks. Atoms bond in molecular graphs.
> GNNs are the natural architecture for learning on these structures.

---
## Step 1 — Motivation

A PPI network: 10,000 nodes (proteins), 200,000 edges (interactions).
Which proteins are disease-related? An MLP would treat each protein independently,
ignoring network context. A GNN propagates information along edges — the
representation of protein $v$ is informed by its neighbors, their neighbors,
and so on. This is the same insight as in Module 12's network medicine: diseases
cluster in network neighborhoods.

---
## Step 2 — Intuition

**Message passing framework:**
Every GNN follows the same pattern at each layer:
1. **Message:** each neighbor $u$ sends a message to $v$ (a function of $u$'s features)
2. **Aggregate:** collect all messages for $v$ (sum, mean, max)
3. **Update:** combine $v$'s own features with the aggregated messages → new representation

Repeat for $k$ layers → each node has seen information from $k$-hop neighbors.

**Graph Convolutional Network (GCN, Kipf & Welling 2017):**
The simplest GNN:
$$H^{(l+1)} = \sigma(\hat{A} H^{(l)} W^{(l)})$$
where $\hat{A} = \tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2}$
(symmetrically normalized adjacency with self-loops).

**Graph-level vs. node-level vs. edge-level tasks:**
- Node classification: is this protein a disease gene?
- Graph classification: is this molecule toxic?
- Edge prediction: do these two proteins interact?
- Link prediction: predict missing edges (drug-target pairs)

---
## Step 3 — Biological Background

**GNNs for PPI-based disease gene discovery:**
Muzio et al. (2021): GCN on STRING PPI network outperforms network propagation
methods for disease gene prioritization. Node features: expression, conservation, domain.

**Molecular GNNs for drug discovery:**
- Atoms = nodes (element, charge, hybridization as features)
- Bonds = edges (bond type, ring membership as features)
- Task: predict logP, solubility, toxicity, binding affinity
- ChemProp (Yang 2019): message-passing NN achieves SOTA on MoleculeNet benchmark

**DiffDock (Corso 2022):**
GNN-based protein-ligand docking. Represents the pocket and ligand as graphs.
Uses SE(3)-equivariant GNN (respects 3D rotational symmetry).

**AlphaFold structure module:**
The backbone geometry update in AlphaFold uses invariant point attention —
a form of attention that respects 3D rotational equivariance.
The residue pair representations form a complete graph (all pairs attended to).

---
## Step 4 — Mathematical Explanation

**GCN layer derivation:**

Add self-loops: $\tilde{A} = A + I$

Degree matrix: $\tilde{D}_{ii} = \sum_j \tilde{A}_{ij}$

Normalized adjacency: $\hat{A} = \tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2}$

GCN update: $H^{(l+1)} = \sigma(\hat{A} H^{(l)} W^{(l)})$

**For node $v$, this computes:**
$$h_v^{(l+1)} = \sigma\!\left(W^{(l)} \sum_{u \in \mathcal{N}(v) \cup \{v\}} \frac{h_u^{(l)}}{\sqrt{\tilde{d}_u \tilde{d}_v}}\right)$$

The $1/\sqrt{\tilde{d}_u \tilde{d}_v}$ factor normalizes by node degrees —
hub nodes (high degree) contribute less per edge than leaf nodes.

**Graph-level pooling (readout):**
After $k$ GCN layers, aggregate all node representations:
$$h_G = \text{READOUT}(\{h_v^{(k)} : v \in G\})$$
Options: mean pooling, sum pooling, attention pooling (DIFFPOOL).

In [ ]:
# Step 6 — Python: GCN from scratch + PyTorch Geometric (fallback: manual)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.metrics import roc_auc_score

torch.manual_seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Build synthetic PPI graph ----
# Use Barabasi-Albert model from Module 12
rng = np.random.default_rng(42)
n_nodes = 300

def barabasi_albert(n, m=3, seed=42):
    rng_ba = np.random.default_rng(seed)
    edges = set()
    stubs = list(range(m*2))  # initial complete graph stub list
    for i in range(m, m+m):
        for j in range(i):
            edges.add((min(i,j), max(i,j))); stubs.extend([i,j])
    for new_node in range(m*2, n):
        targets = set()
        while len(targets) < m:
            candidate = stubs[rng_ba.integers(len(stubs))]
            if candidate != new_node: targets.add(candidate)
        for t in targets:
            edges.add((min(new_node,t), max(new_node,t)))
            stubs.extend([new_node, t])
    return list(edges)

edge_list = barabasi_albert(n_nodes, m=3)
G = nx.Graph()
G.add_nodes_from(range(n_nodes))
G.add_edges_from(edge_list)
print(f'PPI graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# Node features: degree, clustering coeff, betweenness (approximated), random bio features
degrees = dict(G.degree())
clustering = nx.clustering(G)
node_feats = np.array([[degrees[v]/50, clustering[v]] + rng.normal(0,1,8).tolist()
                         for v in range(n_nodes)], dtype=np.float32)  # (n, 10)
print(f'Node features shape: {node_feats.shape}')

# Labels: top-20 hub nodes = "disease hubs" (node classification task)
hub_threshold = sorted(degrees.values())[-20]
y_labels = np.array([1 if degrees[v] >= hub_threshold else 0 for v in range(n_nodes)])
print(f'Positive (hub) nodes: {y_labels.sum()} ({y_labels.mean():.1%})')

# Train/val/test mask
idx = np.arange(n_nodes)
train_mask = np.zeros(n_nodes, dtype=bool)
val_mask = np.zeros(n_nodes, dtype=bool)
test_mask = np.zeros(n_nodes, dtype=bool)
train_idx = rng.choice(idx, 150, replace=False)
rem_idx = np.setdiff1d(idx, train_idx)
val_idx = rng.choice(rem_idx, 75, replace=False)
test_idx = np.setdiff1d(rem_idx, val_idx)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

# ---- GCN from scratch (numpy-based normalized adjacency) ----
def build_normalized_adj(edge_list, n):
    """Build symmetrically normalized adjacency with self-loops."""
    # Adjacency matrix with self-loops
    A = np.zeros((n, n), dtype=np.float32)
    for u, v in edge_list:
        A[u, v] = A[v, u] = 1.0
    np.fill_diagonal(A, 1.0)  # self-loops
    # Degree matrix
    D = A.sum(axis=1)  # (n,)
    D_inv_sqrt = 1.0 / np.sqrt(np.maximum(D, 1e-8))
    # Normalize: D^{-1/2} A D^{-1/2}
    A_norm = D_inv_sqrt[:, np.newaxis] * A * D_inv_sqrt[np.newaxis, :]
    return torch.tensor(A_norm)

A_norm = build_normalized_adj(edge_list, n_nodes).to(device)
X_feat = torch.tensor(node_feats).to(device)
y_torch = torch.tensor(y_labels, dtype=torch.long).to(device)

# ---- GCN model ----
class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3):
        super().__init__()
        self.W1 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.W2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W3 = nn.Linear(hidden_dim, out_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, X, A_norm):
        # Layer 1: A_norm @ X @ W1
        H1 = F.relu(self.W1(A_norm @ X))
        H1 = self.dropout(H1)
        # Layer 2
        H2 = F.relu(self.W2(A_norm @ H1))
        H2 = self.dropout(H2)
        # Layer 3 (output)
        out = self.W3(A_norm @ H2)
        return out  # (n, out_dim)

model_gcn = GCN(in_dim=10, hidden_dim=32, out_dim=2).to(device)
print(f'GCN parameters: {sum(p.numel() for p in model_gcn.parameters()):,}')

# ---- Baseline: MLP (no graph structure) ----
class MLPBaseline(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )
    def forward(self, X, A_norm=None): return self.net(X)

mlp_baseline = MLPBaseline(10, 32, 2).to(device)

# ---- Training function ----
def train_node_clf(model, n_epochs=150):
    opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor([0.1, 0.9]).to(device))  # imbalanced
    train_losses, val_aurocs = [], []
    for epoch in range(n_epochs):
        model.train()
        opt.zero_grad()
        logits = model(X_feat, A_norm)
        loss = criterion(logits[train_mask], y_torch[train_mask])
        loss.backward(); opt.step()
        train_losses.append(loss.item())
        
        model.eval()
        with torch.no_grad():
            probs = F.softmax(logits[val_mask], dim=-1)[:, 1].cpu().numpy()
            auc = roc_auc_score(y_labels[val_mask], probs)
        val_aurocs.append(auc)
    return train_losses, val_aurocs

print('Training GCN...')
gcn_losses, gcn_aurocs = train_node_clf(model_gcn)
print('Training MLP (no graph)...')
mlp_losses, mlp_aurocs = train_node_clf(mlp_baseline)

print(f'GCN final val AUROC: {max(gcn_aurocs):.4f}')
print(f'MLP final val AUROC: {max(mlp_aurocs):.4f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Panel A: Network colored by GCN predictions
ax = axes[0]
model_gcn.eval()
with torch.no_grad():
    logits_all = model_gcn(X_feat, A_norm)
    probs_all = F.softmax(logits_all, dim=-1)[:, 1].cpu().numpy()

pos = nx.spring_layout(G, seed=42, k=0.3)
nodelist = list(G.nodes())
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.15, width=0.3, edge_color='grey')
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=20,
                         node_color=probs_all, cmap='RdYlGn_r', vmin=0, vmax=1)
ax.set_title('A. GCN predicted hub probability\n(green=low risk, red=high risk)')
ax.axis('off')

# Panel B: Val AUROC: GCN vs. MLP
ax = axes[1]
ax.plot(gcn_aurocs, 'tomato', lw=1.5, label=f'GCN (max={max(gcn_aurocs):.3f})')
ax.plot(mlp_aurocs, 'steelblue', lw=1.5, label=f'MLP (max={max(mlp_aurocs):.3f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val AUROC')
ax.set_title('B. GCN vs. MLP\n(GCN uses graph structure)')
ax.legend(fontsize=9)
ax.set_ylim(0.5, 1.0)

# Panel C: Node embeddings (GCN hidden layer) — PCA
from sklearn.decomposition import PCA
model_gcn.eval()
with torch.no_grad():
    # Extract intermediate representation
    H1 = F.relu(model_gcn.W1(A_norm @ X_feat))
    H2 = F.relu(model_gcn.W2(A_norm @ H1)).cpu().numpy()
pca_h = PCA(n_components=2)
H2_2d = pca_h.fit_transform(H2)
ax = axes[2]
ax.scatter(H2_2d[y_labels==0, 0], H2_2d[y_labels==0, 1], c='steelblue', s=20, alpha=0.5, label='Non-hub')
ax.scatter(H2_2d[y_labels==1, 0], H2_2d[y_labels==1, 1], c='tomato', s=60, alpha=0.8,
             zorder=5, label='Hub (disease)', edgecolors='black')
ax.set_title('C. GCN node embeddings (PCA)\n(hub nodes separated in learned space)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(fontsize=8)

# Panel D: Degree vs. predicted probability (GCN leverages topology)
ax = axes[3]
deg_arr = np.array([degrees[v] for v in range(n_nodes)])
ax.scatter(deg_arr, probs_all, alpha=0.4, s=20,
             c=['tomato' if y == 1 else 'steelblue' for y in y_labels])
ax.set_xlabel('Node degree')
ax.set_ylabel('GCN predicted hub probability')
ax.set_title('D. Degree vs. predicted probability\n(GCN learns degree is the key signal)')
from matplotlib.patches import Patch
ax.legend([Patch(color='tomato', label='True hub'), Patch(color='steelblue', label='Non-hub')], fontsize=8)

plt.suptitle('Module 14 NB07: Graph Neural Networks', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('graph_neural_networks.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement one GCN layer from scratch as a matrix multiplication: compute
   $H^{(1)} = \sigma(\hat{A} H^{(0)} W)$ and verify it matches the `GCN.forward` output.
2. Vary the number of GCN layers from 1 to 5. At which depth does performance
   peak? What is the "oversmoothing" problem that appears at many layers?
3. Replace the synthetic task (hub detection) with community detection:
   use the Module 12 SBM planted partition graph and predict community membership.
4. Add edge features: for each edge, add the absolute degree difference of the
   two endpoints as a feature. How would you incorporate edge features into the
   message passing framework?

---
## Step 10 — Quiz

1. Write the GCN layer update equation. What does $\hat{A}$ represent?
2. Why do we add self-loops to the adjacency matrix in GCN? What is the effect
   on the message passing equation?
3. Explain the message-passing framework in 3 steps: message, aggregate, update.
4. What is oversmoothing in deep GNNs? At what layer count does it typically appear?
5. Name three biological tasks where GNNs are used and explain why graph structure
   is important for each.

---
## Step 12 — Reflection

> *[In one paragraph: explain why a GCN with $k$ layers considers $k$-hop
> neighborhoods, and connect this to the network medicine concept of disease
> modules from Module 12 — how does the GCN implicitly perform module-aware
> classification?]*

---
*Next: `08_gnns_for_molecular_biology.ipynb`*